In [13]:
import json

findings = {
    "cleaned_total_revenue_inr": 97358.3,
    "raw_total_revenue_inr": 99860.2,
    "duplicate_reconciliation_delta_inr": 2501.9,
    "return_rate_by_payment": {
        "COD": 44.4,
        "CARD": 14.7,
        "UPI": 18.9
    },
    "highest_risk_segment": {
        "payment_method": "COD",
        "city_tier": 2,
        "return_rate_pct": 54.5
    },
    "true_peak_month": {
        "month": "2026-03",
        "revenue_inr": 20318.9
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": 29582.1,
        "corrected_revenue_inr": 11637.1
    }
}

with open("findings.json", "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2, ensure_ascii=False)

print("findings.json created successfully!")

findings.json created successfully!


In [15]:
import json
import os
from google import genai


def generate_scr_narrative(findings: dict) -> dict:

    system_instruction = """
You are a senior data analyst writing for Mamaearth's regional ops and finance heads.

Write a concise business narrative with exactly three labelled sections:

Situation
Complication
Resolution

Rules:
- Every number must come from the supplied findings.
- Use exactly the same values provided.
- Do not invent statistics, percentages, revenue figures, months, or trends.
- Base all conclusions only on the supplied findings.
"""

    user_prompt = f"""
Use the following verified findings.

Cleaned Revenue: ₹{findings["cleaned_total_revenue_inr"]}
Raw Revenue: ₹{findings["raw_total_revenue_inr"]}
Duplicate Reconciliation Delta: ₹{findings["duplicate_reconciliation_delta_inr"]}

Return Rate by Payment:
{findings["return_rate_by_payment"]}

Highest Risk Segment:
{findings["highest_risk_segment"]}

True Peak Month:
{findings["true_peak_month"]}

Outlier Inflated Month:
{findings["outlier_inflated_month"]}

Create an SCR narrative.
"""

    api_key = os.getenv("GEMINI_API_KEY")

    # Try Gemini if API key is available
    if api_key:

        try:
            client = genai.Client(api_key=api_key)

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=user_prompt,
                config={
                    "system_instruction": system_instruction
                }
            )

            return {
                "narrative": response.text,
                "source": "gemini"
            }

        except Exception:
            pass

    # Keyless fallback
    fallback_narrative = f"""Situation

Cleaned revenue is ₹{findings["cleaned_total_revenue_inr"]}, compared with raw revenue of ₹{findings["raw_total_revenue_inr"]}. Return rates are COD at {findings["return_rate_by_payment"]["COD"]}%, CARD at {findings["return_rate_by_payment"]["CARD"]}%, and UPI at {findings["return_rate_by_payment"]["UPI"]}%.

Complication

The duplicate reconciliation delta is ₹{findings["duplicate_reconciliation_delta_inr"]}. The highest-risk segment is {findings["highest_risk_segment"]["payment_method"]} in city tier {findings["highest_risk_segment"]["city_tier"]}, with a return rate of {findings["highest_risk_segment"]["return_rate_pct"]}%. The apparent revenue for {findings["outlier_inflated_month"]["month"]} was ₹{findings["outlier_inflated_month"]["apparent_revenue_inr"]}, while corrected revenue was ₹{findings["outlier_inflated_month"]["corrected_revenue_inr"]}.

Resolution

Use the cleaned revenue of ₹{findings["cleaned_total_revenue_inr"]} for reporting. The true peak month was {findings["true_peak_month"]["month"]}, with revenue of ₹{findings["true_peak_month"]["revenue_inr"]}. Review the highest-risk segment as part of operational follow-up."""

    return {
        "narrative": fallback_narrative,
        "source": "keyless_fallback"
    }

In [18]:
import os

os.makedirs("narrator/output", exist_ok=True)

print("Folder created successfully!")
print(os.listdir("narrator"))

Folder created successfully!
['output']


In [19]:
import os

print(os.path.exists("findings.json"))

True


In [20]:
import json

with open("findings.json", "r", encoding="utf-8") as f:
    findings = json.load(f)

result = generate_scr_narrative(findings)

with open("narrator/output/narrative.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(result["narrative"])
print("\nSource:", result["source"])

Situation

Cleaned revenue is ₹97358.3, compared with raw revenue of ₹99860.2. Return rates are COD at 44.4%, CARD at 14.7%, and UPI at 18.9%.

Complication

The duplicate reconciliation delta is ₹2501.9. The highest-risk segment is COD in city tier 2, with a return rate of 54.5%. The apparent revenue for 2026-01 was ₹29582.1, while corrected revenue was ₹11637.1.

Resolution

Use the cleaned revenue of ₹97358.3 for reporting. The true peak month was 2026-03, with revenue of ₹20318.9. Review the highest-risk segment as part of operational follow-up.

Source: keyless_fallback


In [21]:
import os

print(os.path.exists("narrator/output/narrative.json"))

True


In [22]:
import os

file_path = "narrator/output/narrative.json"

if os.path.exists(file_path):
    print("✅ narrative.json exists!")
else:
    print("❌ narrative.json NOT found")

✅ narrative.json exists!


In [23]:
import json

with open("narrator/output/narrative.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "narrative": "Situation\n\nCleaned revenue is ₹97358.3, compared with raw revenue of ₹99860.2. Return rates are COD at 44.4%, CARD at 14.7%, and UPI at 18.9%.\n\nComplication\n\nThe duplicate reconciliation delta is ₹2501.9. The highest-risk segment is COD in city tier 2, with a return rate of 54.5%. The apparent revenue for 2026-01 was ₹29582.1, while corrected revenue was ₹11637.1.\n\nResolution\n\nUse the cleaned revenue of ₹97358.3 for reporting. The true peak month was 2026-03, with revenue of ₹20318.9. Review the highest-risk segment as part of operational follow-up.",
  "source": "keyless_fallback"
}


In [24]:
narrative = data["narrative"]

print("Situation:", "Situation" in narrative)
print("Complication:", "Complication" in narrative)
print("Resolution:", "Resolution" in narrative)

Situation: True
Complication: True
Resolution: True
